In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Permite importar tu paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))


# 03 · Preprocesamiento — de las decisiones a código con pruebas

> **Esqueleto.** Guía completa: `curso/kit-estudiante/guia-eda-y-preprocesamiento.md`, sección 4.

Cada decisión de la tabla del EDA se convierte en una **función pequeña** en
`src/proyecto/data/` y en **al menos una prueba** en `tests/`. Este notebook **solo orquesta**:
llama a las funciones y muestra resultados. No contiene lógica reutilizable.

Qué produce: datos limpios, **partición por la unidad de análisis** y una prueba que la verifica.

## 0. Cargar los datos crudos

In [ ]:
import pandas as pd

RAW_DIR = Path.cwd().resolve().parent / "data" / "raw"
files = sorted(p for p in RAW_DIR.glob("*") if p.suffix.lower() in {".csv", ".parquet", ".xlsx"})

if not files:
    df = None
    print("Todavía no hay datos. Coloca tu dataset en data/raw/ (csv, parquet o xlsx) y vuelve a ejecutar.")
else:
    path = files[0]
    df = pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_excel(path) if path.suffix == ".xlsx" else pd.read_csv(path, low_memory=False)
    print(f"{path.name}: {df.shape[0]:,} filas x {df.shape[1]} columnas")
    # TODO: si el dataset trae datos personales (nombres, documentos, teléfonos), elimínalos aquí, antes de mostrar nada
    # df = df.drop(columns=["<columna_personal>"])

# TODO: completa estos nombres cuando conozcas tu dataset
ID_COL = None      # columna que identifica a la entidad (cliente, paciente, empresa...)
TARGET = None      # columna objetivo

## 1. Limpieza

Importa tus funciones de `src/proyecto/data/cleaning.py` (créalas con Claude a partir de tu tabla
de decisiones; renombra el paquete `proyecto` si quieres). **No modifiques `data/raw/`.**

In [ ]:
try:
    from proyecto.data.cleaning import clean   # TODO: crea esta función en src/proyecto/data/cleaning.py
except ModuleNotFoundError:
    clean = None
    print("TODO: crea src/proyecto/data/cleaning.py con una función clean(df) que aplique tus decisiones del EDA.")

if df is None or clean is None:
    clean_df = None
else:
    clean_df = clean(df)
    print(f"antes: {df.shape}  ->  después: {clean_df.shape}")

## 2. Comprobaciones de la limpieza

Convierte tus decisiones en **afirmaciones** que fallan si algo no se cumple (no es un gráfico).

In [ ]:
if clean_df is None:
    print("Sin limpieza todavía: esta celda se ejecutará cuando exista clean(df).")
else:
    checks = {
        "no se perdieron filas": len(clean_df) == len(df),
        # TODO: una comprobación por decisión del EDA, por ejemplo:
        # "sin datos personales": not {"nombre", "documento"} & set(clean_df.columns),
        # "la columna X ya es numérica": pd.api.types.is_numeric_dtype(clean_df["X"]),
    }
    result = pd.Series(checks, name="se cumple")
    assert result.all(), result[~result]
    print(result)

## 3. Partición por la unidad de análisis

Si una entidad aparece en varias filas, **todas sus filas van al mismo lado**. Importa tu función
de `src/proyecto/data/split.py` (modelo en la sección 4.3 de la guía) y **verifica** que ninguna
entidad quede en los dos lados.

In [ ]:
try:
    from proyecto.data.split import split_by_group   # TODO: crea esta función en src/proyecto/data/split.py
except ModuleNotFoundError:
    split_by_group = None
    print("TODO: crea src/proyecto/data/split.py con split_by_group(df, group_col, test_size, seed).")

if clean_df is None or split_by_group is None or ID_COL is None:
    train = test = None
    print("Pendiente: necesitas datos limpios, split_by_group y ID_COL definido.")
else:
    train, test = split_by_group(clean_df, group_col=ID_COL, test_size=0.2)
    assert set(train[ID_COL]).isdisjoint(set(test[ID_COL])), "hay entidades en los dos lados"
    print(f"entrenamiento: {len(train):,} filas · prueba: {len(test):,} filas (cerrada hasta la etapa 7)")

**Mi unidad de análisis y por qué parto así:** TODO

> Si **no** hay entidades repetidas, partir por fila es válido: explícalo aquí por escrito.

## 4. Guardar los datos de la etapa

Se guardan en `data/processed/` (no se versionan: se regeneran con este notebook). La prueba **no
se vuelve a mirar** hasta la validación final (etapa 7).

In [ ]:
if train is None or test is None:
    print("Sin partición todavía: esta celda guardará los datos cuando exista.")
else:
    import joblib
    out = Path.cwd().resolve().parent / "data" / "processed"
    out.mkdir(parents=True, exist_ok=True)
    joblib.dump(train, out / "train.joblib")
    joblib.dump(test, out / "test.joblib")
    print("guardado en data/processed/")

## 5. Las pruebas automáticas

Desde la raíz del repo:

```bash
pytest -q
```

Debe existir **al menos una prueba por decisión** (`tests/test_cleaning.py`) y la prueba de la
partición (`tests/test_split.py`, modelo en la sección 4.3 de la guía).

## Entregable de esta etapa (hito H1)

- [ ] La limpieza corre de principio a fin desde el repo.
- [ ] La partición respeta mi unidad de análisis y está **verificada con una prueba**.
- [ ] `pytest` en verde.
- [ ] `data/raw/` no cambió.
- [ ] Declaré mi unidad de análisis por escrito.